# 📝 AI Dataset Captioning & Tagging Studio
Bộ công cụ chuyên dụng để dọn dẹp, phân tích và gán nhãn tự động cho tập dữ liệu hình ảnh & video sử dụng **Gemini 3.x, Florence-2, JoyCaption, OpenAI**.

### ☕ Bước 1: Cài đặt Môi trường & Key Vault

In [ ]:
# @title ⚙️ 1. Cài đặt Môi trường
import os
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!pip install -q google-genai openai transformers accelerate pillow tqdm

if not os.path.exists('/content/TranningLoras'):
    !git clone https://github.com/nguyenducvuongg/TranningLoras.git /content/TranningLoras
else:
    !git -C /content/TranningLoras pull

%cd /content/TranningLoras
!pip install -q -e .

from lora_trainer.caption.key_manager import display_key_vault_dashboard
print("✅ Đã cài đặt hoàn tất!")
display_key_vault_dashboard()

### 🔐 (Tùy chọn) Quản lý API Key Vault

In [ ]:
# @title 🔐 Quản lý API Key Vault (Thêm / Đổi Key)
Platform = "gemini" # @param ["gemini", "huggingface", "wandb", "openai", "civitai"]
New_API_Key = "" # @param {type:'string'}
Key_Label = "" # @param {type:'string'}

from lora_trainer.caption.key_manager import save_api_key, display_key_vault_dashboard
if New_API_Key.strip():
    save_api_key(Platform, New_API_Key, label=Key_Label or None, set_default=True)
display_key_vault_dashboard()

### 📂 Bước 2: AI Captioning Studio

In [ ]:
# @title ✨ 2. Gán nhãn Tự động
Dataset_Folder = "/content/drive/MyDrive/My_Dataset" # @param {type:'string'}
Caption_Engine = "Gemini-3.6-Flash" # @param ["Gemini-3.6-Flash", "Gemini-3.7-Flash", "Gemini-3.5-Flash", "Gemini-3.5-Flash-Lite", "Gemini-3.1-Pro", "Gemini-3-Pro", "Florence-2", "JoyCaption", "OpenAI-GPT4o"]
Task_Mode = "General" # @param ["General", "Skin_Portrait", "Upscale_Restoration", "Art_Style", "Character_Outfit"]
Caption_Length = "Medium" # @param ["Short", "Medium", "Long"]

# @markdown 🔑 **API Key (Tự động ghi nhớ)**: Để trống nếu muốn dùng Key đã lưu trong Vault.
API_Key = "" # @param {type:'string'}
Overwrite_Existing = False # @param {type:'boolean'}
Is_Video_Dataset = False # @param {type:'boolean'}

from lora_trainer.caption.gemini_captioner import batch_caption_gemini
from lora_trainer.caption.florence_captioner import batch_caption_florence
from lora_trainer.caption.joy_captioner import batch_caption_joycaption
from lora_trainer.caption.openai_captioner import batch_caption_openai

if Caption_Engine.startswith("Gemini"):
    batch_caption_gemini(
        Dataset_Folder,
        api_key=API_Key,
        model_alias=Caption_Engine,
        length_preset=Caption_Length,
        task_mode=Task_Mode,
        overwrite=Overwrite_Existing,
        is_video_folder=Is_Video_Dataset,
    )
elif Caption_Engine == "Florence-2":
    batch_caption_florence(Dataset_Folder, task_preset=Caption_Length, overwrite=Overwrite_Existing)
elif Caption_Engine == "JoyCaption":
    batch_caption_joycaption(Dataset_Folder, caption_length=Caption_Length.lower(), overwrite=Overwrite_Existing)
elif Caption_Engine == "OpenAI-GPT4o":
    batch_caption_openai(Dataset_Folder, api_key=API_Key, overwrite=Overwrite_Existing)

### 🏷️ Bước 3: Thao tác Tag Nâng Cao (Thêm/Sửa/Xóa Trigger Words)

In [ ]:
# @title 🏷️ 3. Quản lý Thẻ Tag & Trigger Word
Dataset_Folder = "/content/drive/MyDrive/My_Dataset" # @param {type:'string'}
Trigger_Word = "" # @param {type:'string'}
Action = "Prepend" # @param ["Prepend", "Append", "Remove", "Add_Folder_Name"]

from lora_trainer.data.tag_processor import process_dir_tags, add_folder_name_tags

if Action == "Prepend":
    process_dir_tags(Dataset_Folder, Trigger_Word, append=False)
elif Action == "Append":
    process_dir_tags(Dataset_Folder, Trigger_Word, append=True)
elif Action == "Remove":
    process_dir_tags(Dataset_Folder, Trigger_Word, remove_tag=True)
elif Action == "Add_Folder_Name":
    add_folder_name_tags(Dataset_Folder)

print("✅ Đã cập nhật xong toàn bộ tag!")

### 🔢 Bước 4: Chuẩn Hóa Tên Tệp Dataset Tự Động (Renamer Tool)
💡 Tự động quét và đổi tên toàn bộ ảnh và file `.txt` caption trong thư mục thành dạng chuẩn (VD: `0001.png`, `0001.txt` hoặc `char_0001.jpg`, `char_0001.txt`), đồng thời đồng bộ thư mục Control nếu có.

In [ ]:
# @title 🔢 4. Chuẩn Hóa & Đổi Tên Tệp Hàng Loạt
Train_Folders = "/content/drive/MyDrive/My_Dataset" # @param {type:'string'}
Control_Folder = "" # @param {type:'string'}
Filename_Prefix = "" # @param {type:'string'}
Number_Of_Digits = 4 # @param {type:'integer'}
Auto_Create_Missing_Captions = True # @param {type:'boolean'}
Default_Caption_Text = "" # @param {type:'string'}

from lora_trainer.data.renamer import batch_standardize_datasets

batch_standardize_datasets(
    train_folders=Train_Folders,
    control_folders=Control_Folder if Control_Folder else None,
    prefix=Filename_Prefix,
    digits=Number_Of_Digits,
    auto_create_txt=Auto_Create_Missing_Captions,
    default_caption=Default_Caption_Text,
)